In [1]:
!pip install ultralytics easyocr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00a 0:00:01


In [9]:
import os, yaml, shutil, cv2
import numpy as np
from ultralytics import YOLO
import easyocr

# ── Configuration ──────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection'   # <-- CHANGE THIS
PROJECT_NAME = 'license_plate_detection'
MODEL_BASE   = 'yolov8n.pt'
EPOCHS       = 80
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

In [4]:
import yaml

config = {
    'path': '/kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': ['LisencePlate']
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(config, f)

!ls /kaggle/working

data.yaml


In [5]:
data_yaml = '/kaggle/working/data.yaml'

model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,
    patience= 25,
    save    = True,
    plots   = True,
    # License plate: rectangular, needs aspect ratio awareness
    scale   = 0.3,
    translate = 0.05,
    degrees = 3.0,        # plates rarely tilted much
    fliplr  = 0.5,
    mosaic  = 0.3,
    erasing = 0.2,        # simulate partial occlusion
    close_mosaic = 15,
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.3, multi_scale=0.0, name=license_plate_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

In [6]:
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 14.8±3.8 MB/s, size: 19.8 KB)
val: Scanning /kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection/valid/labels... 2048 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2048/2048 429.2it/s 4.8s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 9.7it/s 13.2s0.2ss
                   all       2048       2195      0.983      0.936      0.962      0.658
Speed: 0.7ms preprocess, 2.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP50: 0.9619941412100906
mAP50-95: 0.658155892011

In [7]:
output_path = '/kaggle/working/model4_license_plate.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved: {output_path}')

Model saved: /kaggle/working/model4_license_plate.pt


In [11]:
# ── OCR pipeline test ───────────────────────────────────────────
def preprocess_plate(crop):
    """Enhance plate crop for better OCR accuracy."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    # Upscale for OCR
    h, w = gray.shape
    gray = cv2.resize(gray, (w*2, h*2), interpolation=cv2.INTER_CUBIC)
    # Denoise + sharpen
    gray = cv2.fastNlMeansDenoising(gray, h=10)
    kernel = np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]])
    gray = cv2.filter2D(gray, -1, kernel)
    # Adaptive threshold for variable lighting
    binary = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    return binary

reader = easyocr.Reader(['en'], gpu=True)
print('EasyOCR reader initialised')

def read_plate(image_path, plate_model, ocr_reader):
    """Full pipeline: detect plate → crop → OCR."""
    img = cv2.imread(image_path)
    results = plate_model.predict(img, conf=0.4, verbose=False)
    plates = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            crop = img[y1:y2, x1:x2]
            proc = preprocess_plate(crop)
            text_results = ocr_reader.readtext(
                proc,
                allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                detail=1
            )
            plate_text = ''.join([
                t[1] for t in text_results if t[2] > 0.4
            ]).strip()
            plates.append({
                'bbox': (x1, y1, x2, y2),
                'confidence': float(box.conf),
                'plate_number': plate_text
            })
    return plates

# Test on a sample
import glob, random
samples = glob.glob(os.path.join(DATASET_PATH, 'valid/images/*.jpg'))[:1]
if samples:
    test_img = random.choice(samples)
    detected = read_plate(test_img, model_best, reader)
    print('Test image:', test_img)
    for d in detected:
        print(f'  Plate: {d["plate_number"]}  conf={d["confidence"]:.2f}')
else:
    print('No val images found for test')

EasyOCR reader initialised
Test image: /kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection/valid/images/CarLongPlateGen2256_jpg.rf.fdb57b471473e661eb9bbecdcde15eae.jpg
  Plate:   conf=0.81


In [27]:
import cv2
import numpy as np
import easyocr
import os

# ── Preprocessing (FIXED) ──────────────────────────────────────
def preprocess_plate(crop):
    """Light preprocessing (OCR-friendly)."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    
    # Upscale (important for OCR)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    
    # Light denoise (not aggressive)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    
    return gray


# ── Initialize OCR ─────────────────────────────────────────────
reader = easyocr.Reader(['en'], gpu=True)
print('EasyOCR reader initialised')


# ── OCR Pipeline ───────────────────────────────────────────────
def read_plate(image_path, plate_model, ocr_reader):
    """Detect plate → crop → OCR."""
    
    img = cv2.imread(image_path)
    H, W, _ = img.shape
    
    results = plate_model.predict(img, conf=0.4, verbose=False)
    plates = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # 🔥 Expand bounding box slightly
            pad = 5
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(W, x2 + pad)
            y2 = min(H, y2 + pad)

            crop = img[y1:y2, x1:x2]

            # Try BOTH raw + processed (best practice)
            proc = preprocess_plate(crop)

            # OCR on processed image
            text_results = ocr_reader.readtext(
                proc,
                allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                detail=1
            )

            # 🔥 DEBUG: print raw OCR output
            print("OCR raw:", text_results)

            # 🔥 Combine all detected text (no strict filtering)
            plate_text = ' '.join([t[1] for t in text_results]).strip()

            plates.append({
                'bbox': (x1, y1, x2, y2),
                'confidence': float(box.conf),
                'plate_number': plate_text
            })

    return plates


# ── Test on sample ─────────────────────────────────────────────
import glob, random

samples = glob.glob(os.path.join(DATASET_PATH, 'test/images/*.jpg'))[:5]

if samples:
    test_img = random.choice(samples)
    detected = read_plate(test_img, model_best, reader)

    print('Test image:', test_img)

    for d in detected:
        print(f'Plate: {d["plate_number"]} | conf={d["confidence"]:.2f}')
else:
    print('No validation images found')

EasyOCR reader initialised
OCR raw: [([[np.int32(45), np.int32(33)], [np.int32(163), np.int32(33)], [np.int32(163), np.int32(97)], [np.int32(45), np.int32(97)]], '66P1', np.float64(0.8503181338310242)), ([[np.int32(27), np.int32(87)], [np.int32(179), np.int32(87)], [np.int32(179), np.int32(155)], [np.int32(27), np.int32(155)]], '51993', np.float64(0.9999610821396641))]
Test image: /kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection/test/images/xemay957_jpg.rf.1aa49b73103439367caa82d9c6327358.jpg
Plate: 66P1 51993 | conf=0.88


In [32]:
# ── FULL OCR PIPELINE (ONE CELL) ───────────────────────────────

import cv2
import numpy as np
import easyocr
import os
import re
import torch
import random

# Fix randomness (stability)
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# ── Preprocessing ─────────────────────────────────────────────
def preprocess_plate(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    return gray

# ── OCR extraction ────────────────────────────────────────────
def ocr_plate(crop, reader):
    proc = preprocess_plate(crop)

    results = reader.readtext(proc, detail=1)

    # sort left → right
    results = sorted(results, key=lambda x: x[0][0][0])

    # combine text
    text = ''.join([r[1] for r in results])

    return text

# ── Cleaning ──────────────────────────────────────────────────
def clean_text(text):
    text = text.replace(" ", "").upper()

    # fix common OCR mistakes
    text = text.replace("O", "0")
    text = text.replace("I", "1")
    text = text.replace("Z", "2")
    text = text.replace("S", "5")
   # text = text.replace("L", "4")

    return text

# ── Validation (Indian plate format) ──────────────────────────
def validate_plate(text):
    pattern = r'[A-Z]{2}[0-9]{2}[A-Z]{1,2}[0-9]{4}'
    match = re.search(pattern, text)
    return match.group(0) if match else text

# ── Final OCR function ────────────────────────────────────────
def get_plate_text(crop, reader):
    raw_text = ocr_plate(crop, reader)
    cleaned = clean_text(raw_text)
    final = validate_plate(cleaned)
    return final

# ── Initialize OCR ────────────────────────────────────────────
reader = easyocr.Reader(['en'], gpu=True)
print("EasyOCR initialized")

# ── Full pipeline: detect → crop → OCR ────────────────────────
def read_plate(image_path, plate_model, reader):
    img = cv2.imread(image_path)
    H, W, _ = img.shape

    results = plate_model.predict(img, conf=0.4, verbose=False)
    outputs = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # expand bounding box (IMPORTANT)
            pad = 10
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(W, x2 + pad)
            y2 = min(H, y2 + pad)

            crop = img[y1:y2, x1:x2]

            plate_text = get_plate_text(crop, reader)

            outputs.append({
                "bbox": (x1, y1, x2, y2),
                "confidence": float(box.conf),
                "plate_number": plate_text
            })

    return outputs

# ── Test ──────────────────────────────────────────────────────
import glob, random

samples = glob.glob(os.path.join(DATASET_PATH, 'test/images/*.jpg'))[:1]

if samples:
    test_img = random.choice(samples)
    results = read_plate(test_img, model_best, reader)

    print("Test image:", test_img)
    for r in results:
        print(f'Plate: {r["plate_number"]} | conf={r["confidence"]:.2f}')
else:
    print("No validation images found")

EasyOCR initialized
Test image: /kaggle/input/datasets/abhishekpriya/license-plate-dataset/Number_Plate_Detection/test/images/xemay2226_jpg.rf.46aee8ad3062ed22338fff531118ac5e.jpg
Plate: 51-ULL579 | conf=0.84
